In [ ]:
from google.cloud import bigquery
import pandas as pd
import os

PROJECT_ID = "prj-grp-relevance-dev-2867"

client = bigquery.Client(project=PROJECT_ID)

# načtení dat z BigQuery
query = """
SELECT
  query_text, user_region_res5, user_region_res4, label_segment_4
FROM `prj-grp-relevance-dev-2867.radius_train.query_region_radius_training_4class_v1`
"""

df_bq = client.query(query).to_dataframe()
print(f"Načteno z BigQuery: {df_bq.shape[0]} řádků, {df_bq.shape[1]} sloupců")

# Uložení do TSV souboru
os.makedirs("data", exist_ok=True)
df_bq.to_csv("data/train.tsv", sep="\t", index=False)
print(f"Data uložena do data/train.tsv")
print(f"Sloupce: {list(df_bq.columns)}")


In [ ]:
# Použití dat z BigQuery (nebo načtení z TSV, pokud první buňka nebyla spuštěna)
try:
    df = df_bq.copy()
    print(f"Použita data z BigQuery: {df.shape}")
except NameError:
    # Pokud první buňka nebyla spuštěna, načteme z TSV
    df = pd.read_csv("data/train.tsv", sep="\t")
    print(f"Načteno z TSV: {df.shape}")

df.head(100)


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/zphilipp/miniconda3/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(1568, 4)


,query_text,user_region_res5,user_region_res4,label_segment_4
0,jamnola,85444643fffffff,8444465ffffffff,0
1,sams club,8529a1d7fffffff,8429a1dffffffff,3
2,costco,85268cdbfffffff,84268cdffffffff,3
3,san antonio zoo,85489c1bfffffff,84489c1ffffffff,0
4,costco,8529a56ffffffff,8429a57ffffffff,3
5,costco,852664cbfffffff,842664dffffffff,3
6,oil change,852a1343fffffff,842a135ffffffff,0
7,sky zone,852aa8d3fffffff,842aa8dffffffff,0
8,costco,852664cffffffff,842664dffffffff,3
9,massage,8529a18bfffffff,8429a19ffffffff,0


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# df: result from query_region_radius_training_4class_v1
X = df[["query_text", "user_region_res5", "user_region_res4"]]
y = df["label_segment_4"].astype(int)  # 0,1,2,3

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

text_features = "query_text"
cat_features = ["user_region_res5", "user_region_res4"]

preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=5), text_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(
            multi_class="multinomial",
            max_iter=1000,
            n_jobs=-1
        )),
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.83      0.97      0.90       157
           1       0.50      0.06      0.11        16
           2       0.86      0.68      0.76        62
           3       1.00      1.00      1.00        79

    accuracy                           0.88       314
   macro avg       0.80      0.68      0.69       314
weighted avg       0.86      0.88      0.86       314



In [21]:
def predict_radius_segment(query_text, user_region_res5, user_region_res4):
    row = {
        "query_text": str(query_text),
        "user_region_res5": user_region_res5 or "UNKNOWN",
        "user_region_res4": user_region_res4 or "UNKNOWN",
    }
    X_input = pd.DataFrame([row])
    pred_segment = int(model.predict(X_input)[0])
    prob_vec = model.predict_proba(X_input)[0]
    probs = {int(cls): float(p) for cls, p in zip(model.classes_, prob_vec)}
    return {"predicted_segment": pred_segment, "probs": probs}


In [26]:
import pandas as pd
from io import StringIO

all_rows = [
    # --- test_rows ---
    {"query_text": "massage",          "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "masage",           "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "massage",          "user_region_res5": "8544f003fffffff", "user_region_res4": "8444f01ffffffff"},
    {"query_text": "facial",           "user_region_res5": "852b9bc7fffffff", "user_region_res4": "842b9bdffffffff"},
    {"query_text": "smog check",       "user_region_res5": "8528308bfffffff", "user_region_res4": "8428309ffffffff"},
    {"query_text": "oil change",       "user_region_res5": "85269607fffffff", "user_region_res4": "8426961ffffffff"},

    {"query_text": "trampoline park",  "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "escape room",      "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "zoo",              "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "sky zone",         "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "bowling",          "user_region_res5": "85262cd7fffffff", "user_region_res4": "84262cdffffffff"},

    {"query_text": "seaworld",         "user_region_res5": "852986bbfffffff", "user_region_res4": "842986bffffffff"},
    {"query_text": "great wolf lodge", "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "great wolf",       "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "citypass",         "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "whale watching",   "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "hotel",            "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "hotl",             "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},

    {"query_text": "windows 11",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "microsoft office", "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "costco",           "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "cosco",            "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "windows 10",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},

    # --- raw_block parsed ---
    {"query_text": "airport parking", "user_region_res5": "8529b6d3fffffff", "user_region_res4": "8429b6dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664cffffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664cffffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "bared monkey", "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "bible museum", "user_region_res5": "852aa847fffffff", "user_region_res4": "842aa85ffffffff"},
    {"query_text": "bible museum", "user_region_res5": "852aa847fffffff", "user_region_res4": "842aa85ffffffff"},
    {"query_text": "big air", "user_region_res5": "8544da87fffffff", "user_region_res4": "8444da9ffffffff"},
    {"query_text": "big air trampoline park", "user_region_res5": "8544d84ffffffff", "user_region_res4": "8444d85ffffffff"},

    # --- přidaný řádek ---
    {"query_text": "laser hair removal", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
]

# 5) final test DataFrame and predictions
df_test = pd.DataFrame(all_rows)

probs = model.predict_proba(df_test[["query_text", "user_region_res5", "user_region_res4"]])
pred_segments = model.predict(df_test[["query_text", "user_region_res5", "user_region_res4"]])

classes = model.named_steps["clf"].classes_

def probs_row(p):
    return {int(c): float(v) for c, v in zip(classes, p)}

df_out = df_test.copy()
df_out["predicted_segment"] = pred_segments.astype(int)
df_out["probs"] = [probs_row(p) for p in probs]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

df_out.head(1000)


,query_text,user_region_res5,user_region_res4,predicted_segment,probs
0,massage,852a1423fffffff,842a143ffffffff,0,"{0: 0.8848086283099812, 1: 0.07389455328191152, 2: 0.033396669805144226, 3: 0.007900148602962924}"
1,masage,852a1423fffffff,842a143ffffffff,0,"{0: 0.7178254976545523, 1: 0.035513598671021065, 2: 0.16655146911359234, 3: 0.08010943456083437}"
2,massage,8544f003fffffff,8444f01ffffffff,0,"{0: 0.9047361759612925, 1: 0.06130656670769008, 2: 0.02764952323519146, 3: 0.006307734095825863}"
3,facial,852b9bc7fffffff,842b9bdffffffff,0,"{0: 0.910710628492405, 1: 0.013942883722744551, 2: 0.04888094731365734, 3: 0.026465540471192952}"
4,smog check,8528308bfffffff,8428309ffffffff,0,"{0: 0.9524966064925059, 1: 0.009003148799928792, 2: 0.024758224337695683, 3: 0.01374202036986966}"
5,oil change,85269607fffffff,8426961ffffffff,0,"{0: 0.5556177542275449, 1: 0.10939028009105962, 2: 0.32515434381323205, 3: 0.009837621868163468}"
6,trampoline park,852a1073fffffff,842a107ffffffff,0,"{0: 0.8718972155455295, 1: 0.022558560555835487, 2: 0.07335044659173252, 3: 0.03219377730690232}"
7,escape room,8526640ffffffff,8426641ffffffff,0,"{0: 0.7630041738350567, 1: 0.025412740273246942, 2: 0.16298746612744228, 3: 0.0485956197642541}"
8,zoo,8529a54ffffffff,8429a55ffffffff,0,"{0: 0.9191264530899869, 1: 0.005888525681168711, 2: 0.056738999847094414, 3: 0.01824602138175005}"
9,sky zone,8526640ffffffff,8426641ffffffff,0,"{0: 0.7390111836747465, 1: 0.039503006378051535, 2: 0.18201922588729585, 3: 0.03946658405990602}"
